In [1]:
# importing libraries
import os
import glob
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
# =============================================================================
# 1. CONFIGURATION & CONSTANTS
# =============================================================================

LOG_DIR = './log_mo'
OUTPUT_DIR = './plots_mo'

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# keep these exact orders for consistent plotting
TARGET_ALGOS = [
    'NSGA-II_Pure', 'NT_NSGA-II', 'NT_1st_Obj_Elitism', 
    'NT_All_Objs_Elitism', 'NT_Rank_CD_Elitism', 
    'NT_Ideal_Cand_Elitism', 'NT_No_Elitism'
]
# get easy-to-read labels for the algorithms
ALGO_LABELS = {
    'NSGA-II_Pure': 'NSGA-II',
    'NT_NSGA-II': 'NT_NSGA-II',
    'NT_1st_Obj_Elitism': 'NT_1st_Obj',
    'NT_All_Objs_Elitism': 'NT_All_Objs',
    'NT_Rank_CD_Elitism': 'NT_CD',
    'NT_Ideal_Cand_Elitism': 'NT_Ideal_Cand',
    'NT_No_Elitism': 'NT_NE'
}



# NOTE: These are the algorithms that have a "jump" in their smoothed curves (changing every 10 generations)
SMOOTH_ALGOS = ['NSGA-II_Pure', 'NT_Ideal_Cand_Elitism', 'NT_No_Elitism', 'NT_NSGA-II', 'NT_Rank_CD_Elitism']

# Colours to distinguish between test and train curves for each algorithm in the line plots (Evolution and Std Dev)
ALGO_COLORS = {
    'NSGA-II_Pure':          {'test': '#9467bd', 'train': '#c5b0d5'}, 
    'NT_NSGA-II':            {'test': '#2ca02c', 'train': '#98df8a'},
    'NT_1st_Obj_Elitism':    {'test': '#17becf', 'train': '#9edae5'},
    'NT_Rank_CD_Elitism':    {'test': '#ff7f0e', 'train': '#ffbb78'},
    'NT_Ideal_Cand_Elitism': {'test': '#d62728', 'train': '#ff9896'},
    'NT_No_Elitism':         {'test': '#8c564b', 'train': '#c49c94'},
    'NT_All_Objs_Elitism':   {'test': '#e377c2', 'train': '#f7b6d2'}
}

In [3]:
# y-axis limits for the boxplots and line plots (evolution and std dev) for each dataset
LIMITS = {
    'Boston': {
        'boxplot_rmse': [0, 15], 'boxplot_size': [0, 16],
        'evol_rmse': [0, 15],    'evol_size': [0, 15]
    },
    'Cooling': {
        'boxplot_rmse': [0, 13], 'boxplot_size': [0, 15],
        'evol_rmse': [0, 12],    'evol_size': [0, 18]
    },
    'Toxicity': {
        'boxplot_rmse': [0, 3100],  'boxplot_size': [0, 30],
        'evol_rmse': [1250, 3000],  'evol_size': [0, 24]
    }
}


DATASETS_LIST = ['Boston', 'Cooling', 'Toxicity']
OBJECTIVES_LIST = ['2_Objs', '3_Objs', '5_Objs']

# Font settings for all plots NOTE: These can be adjusted later
FONT_FAMILY = "Arial, sans-serif"
SIZE_LABELS = 24
SIZE_TICKS = 20
SIZE_LEGEND = 20


# Set Seaborn theme and font settings for all plots
sns.set_theme(style="whitegrid", rc={
    "font.family": FONT_FAMILY, "axes.labelsize": SIZE_LABELS,       
    "xtick.labelsize": SIZE_TICKS, "ytick.labelsize": SIZE_TICKS,       
    "legend.fontsize": SIZE_LEGEND
})

# Font configuration for Plotly plots (consistent with Seaborn/Matplotlib settings)
PLOTLY_FONT_CONFIG = dict(family=FONT_FAMILY, size=SIZE_TICKS, color="black")

In [4]:
# =============================================================================
# 2. DATA LOADER
# =============================================================================

def parse_fitness_column(series):
    series_str = series.astype(str)
    try: return series_str.str.split('|', expand=True)[0].astype(float)
    except: return None

def load_dataset_data(dataset_name):
    history_dict = {'2_Objs': {}, '3_Objs': {}, '5_Objs': {}}
    final_rows = []
    
    target_path = os.path.join(LOG_DIR, dataset_name)
    if not os.path.exists(target_path): return {}, pd.DataFrame()

    for obj_key in OBJECTIVES_LIST:
        for scen in TARGET_ALGOS:
            pattern = os.path.join(target_path, scen, obj_key, "fold_*", "execution_log.csv")
            files = glob.glob(pattern)
            history_dict[obj_key][scen] = []
            
            for f in files:
                try:
                    df = pd.read_csv(f)
                    if df.empty: continue
                    
                    idx_offset = 4 
                    test_idx, train_idx, size_idx, std_idx = 6+idx_offset, 5+idx_offset, 7+idx_offset, 8+idx_offset
                    
                    full_test = parse_fitness_column(df.iloc[:, test_idx])
                    full_train = parse_fitness_column(df.iloc[:, train_idx])
                    full_size = df.iloc[:, size_idx]
                    full_std = df.iloc[:, std_idx]
                    
                    clean_hist = pd.DataFrame({
                        'Test_RMSE': full_test.values,
                        'Train_RMSE': full_train.values,
                        'Test_Size': full_size.values,
                        'Std_RMSE': full_std.values,
                        'Generation': np.arange(len(full_test))
                    })
                    history_dict[obj_key][scen].append(clean_hist)
                    
                    # Final generation para Boxplots
                    try: fold_num = int(f.split(os.sep)[-2].split('_')[-1])
                    except: fold_num = 0

                    final_rows.append({
                        'Dataset': dataset_name,
                        'Scenario': ALGO_LABELS[scen],
                        'Num_Objs': obj_key.split('_')[0],
                        'Fold': fold_num,
                        'Train_RMSE': full_train.iloc[-1],
                        'Test_RMSE': full_test.iloc[-1],
                        'Test_Size': full_size.iloc[-1]
                    })
                except: pass

    return history_dict, pd.DataFrame(final_rows)

In [5]:
# =============================================================================
# 3. PLOTTING FUNCTIONS
# =============================================================================

In [6]:
def plot_custom_rmse_boxplots(df, dataset_name):
    save_dir = os.path.join(OUTPUT_DIR, dataset_name)
    if not os.path.exists(save_dir): os.makedirs(save_dir)
    
    df_melt = df.melt(id_vars=['Scenario', 'Num_Objs'], value_vars=['Train_RMSE', 'Test_RMSE'], var_name='Type', value_name='RMSE')
    df_melt['Hue_Group'] = df_melt['Num_Objs'] + "_" + df_melt['Type']
    
    palette = {
        '2_Train_RMSE': '#aec7e8', '2_Test_RMSE': '#1f77b4', # 2 Objs: Azuis
        '3_Train_RMSE': '#ffbb78', '3_Test_RMSE': '#ff7f0e', # 3 Objs: Laranjas
        '5_Train_RMSE': '#98df8a', '5_Test_RMSE': '#2ca02c'  # 5 Objs: Verdes
    }
    order = [ALGO_LABELS[algo] for algo in TARGET_ALGOS]
    hue_order = ['2_Train_RMSE', '2_Test_RMSE', '3_Train_RMSE', '3_Test_RMSE', '5_Train_RMSE', '5_Test_RMSE']

    plt.figure(figsize=(16, 7))
    ax = sns.boxplot(data=df_melt, x='Scenario', y='RMSE', hue='Hue_Group', order=order, hue_order=hue_order, 
                     palette=palette, width=0.8, linewidth=1.5, showfliers=False)
    
    plt.ylim(LIMITS[dataset_name]['boxplot_rmse'])
    plt.xlabel(''); plt.ylabel('RMSE')
    
    # Custom Legend on TOP
    handles, labels = ax.get_legend_handles_labels()
    clean_labels = [l.replace('_RMSE', '').replace('_', ' Objs (') + ')' for l in labels]
    plt.legend(handles, clean_labels, bbox_to_anchor=(0.5, 1.15), loc='center', ncol=3, frameon=False)
    
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"X_Boxplot_RMSE_{dataset_name}.png"), dpi=300, bbox_inches='tight')
    plt.close()

In [7]:
def plot_custom_size_boxplots(df, dataset_name):
    save_dir = os.path.join(OUTPUT_DIR, dataset_name)
    
    palette = {'2': '#1f77b4', '3': '#ff7f0e', '5': '#2ca02c'}
    order = [ALGO_LABELS[algo] for algo in TARGET_ALGOS]

    plt.figure(figsize=(14, 7))
    ax = sns.boxplot(data=df, x='Scenario', y='Test_Size', hue='Num_Objs', order=order, 
                     palette=palette, width=0.6, linewidth=1.5, showfliers=False)
    
    plt.ylim(LIMITS[dataset_name]['boxplot_size'])
    plt.xlabel(''); plt.ylabel('Size (Nodes)')
    
    handles, labels = ax.get_legend_handles_labels()
    clean_labels = [l + " Objs" for l in labels]
    plt.legend(handles, clean_labels, bbox_to_anchor=(0.5, 1.1), loc='center', ncol=3, frameon=False)
    
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"X_Boxplot_Size_{dataset_name}.png"), dpi=300, bbox_inches='tight')
    plt.close()

In [8]:
def plot_custom_evolution(history_dict, dataset_name, metric_type='RMSE'):
    save_dir = os.path.join(OUTPUT_DIR, dataset_name)
    fig = make_subplots(rows=1, cols=3, shared_yaxes=False, horizontal_spacing=0.05, 
                        subplot_titles=["2 Objectives", "3 Objectives", "5 Objectives"])
    
    y_min, y_max = LIMITS[dataset_name][f'evol_{metric_type.lower()}']

    for col_idx, obj_key in enumerate(['2_Objs', '3_Objs', '5_Objs']):
        col = col_idx + 1
        for algo in TARGET_ALGOS:
            runs = history_dict[obj_key].get(algo, [])
            if not runs: continue
            
            # --- logic for dynamic smoothing (applied to both RMSE and Size) ---
            step = 10 if algo in SMOOTH_ALGOS else 1
            
            min_len = min([len(r) for r in runs])
            col_name = 'Test_RMSE' if metric_type == 'RMSE' else 'Test_Size'
            
            data_test = np.array([r[col_name].values[:min_len] for r in runs])
            median_test = np.median(data_test, axis=0)[::step]
            q1_test = np.percentile(data_test, 15, axis=0)[::step]
            q3_test = np.percentile(data_test, 85, axis=0)[::step]
            generations = np.arange(min_len)[::step]
            
            color_test = ALGO_COLORS[algo]['test']
            label = ALGO_LABELS[algo]
            show_leg = True if col == 1 else False
            
            # Shaded Area
            fig.add_trace(go.Scatter(
                x=np.concatenate([generations, generations[::-1]]),
                y=np.concatenate([q3_test, q1_test[::-1]]),
                fill='toself', fillcolor=color_test, opacity=0.1,
                line=dict(width=0), showlegend=False, hoverinfo='skip'
            ), row=1, col=col)

            # Solid Line (Test)
            fig.add_trace(go.Scatter(
                x=generations, y=median_test, mode='lines',
                line=dict(color=color_test, width=3.5), name=label,
                legendgroup=algo, showlegend=show_leg
            ), row=1, col=col)

            # Dashed Line (Train) - obviously only for RMSE
            if metric_type == 'RMSE':
                data_train = np.array([r['Train_RMSE'].values[:min_len] for r in runs])
                median_train = np.median(data_train, axis=0)[::step]
                fig.add_trace(go.Scatter(
                    x=generations, y=median_train, mode='lines',
                    line=dict(color=ALGO_COLORS[algo]['train'], width=2.5, dash='dash'),
                    legendgroup=algo, showlegend=False 
                ), row=1, col=col)
                
        fig.update_yaxes(range=[y_min, y_max], title_text=metric_type if col==1 else "", row=1, col=col)
        fig.update_xaxes(title_text="Generations", row=1, col=col)

    fig.update_annotations(font=dict(size=SIZE_LABELS, family=FONT_FAMILY))
    fig.update_layout(
        height=600, width=1600, template="plotly_white", font=PLOTLY_FONT_CONFIG,
        margin=dict(t=120, b=80),  # Top margin extra large for legend
        legend=dict(orientation="h", yanchor="bottom", y=1.08, xanchor="center", x=0.5, font=dict(size=SIZE_LEGEND))
    )
    
    fig.write_image(os.path.join(save_dir, f"X_Evolution_{metric_type}_{dataset_name}.png"))

In [9]:
def plot_global_std_dev(all_datasets_history):
    fig = make_subplots(rows=1, cols=3, subplot_titles=DATASETS_LIST, horizontal_spacing=0.08)
    
    for col_idx, ds_name in enumerate(DATASETS_LIST):
        col = col_idx + 1
        history_dict = all_datasets_history.get(ds_name, {})
        if not history_dict: continue

        for algo in TARGET_ALGOS:
            all_runs_std = []
            for obj_key in OBJECTIVES_LIST:
                runs = history_dict.get(obj_key, {}).get(algo, [])
                for r in runs: all_runs_std.append(r['Std_RMSE'].values)
            
            if not all_runs_std: continue
            
            # --- logic for dynamic smoothing (applied to both RMSE and Size) ---
            step = 10 if algo in SMOOTH_ALGOS else 1
            
            min_len = min([len(x) for x in all_runs_std])
            data = np.array([x[:min_len] for x in all_runs_std])
            
            median = np.median(data, axis=0)[::step]
            q1 = np.percentile(data, 15, axis=0)[::step]
            q3 = np.percentile(data, 85, axis=0)[::step]
            generations = np.arange(min_len)[::step]
            
            color = ALGO_COLORS[algo]['test']
            label = ALGO_LABELS[algo]
            show_leg = True if col == 1 else False

            fig.add_trace(go.Scatter(
                x=np.concatenate([generations, generations[::-1]]),
                y=np.concatenate([q3, q1[::-1]]),
                fill='toself', fillcolor=color, opacity=0.1, line=dict(width=0), showlegend=False
            ), row=1, col=col)
            
            fig.add_trace(go.Scatter(
                x=generations, y=median, mode='lines', line=dict(color=color, width=3.5),
                name=label, legendgroup=algo, showlegend=show_leg
            ), row=1, col=col)
        
        fig.update_yaxes(type="log", title_text="RMSE Std Dev" if col==1 else "", row=1, col=col)
        fig.update_xaxes(title_text="Generations", row=1, col=col)

    fig.update_annotations(font=dict(size=SIZE_LABELS, family=FONT_FAMILY))
    fig.update_layout(
        height=600, width=1600, template="plotly_white", font=PLOTLY_FONT_CONFIG,
        margin=dict(t=120, b=80), 
        legend=dict(orientation="h", yanchor="bottom", y=1.08, xanchor="center", x=0.5, font=dict(size=SIZE_LEGEND))
    )
    
    fig.write_image(os.path.join(OUTPUT_DIR, "X_Global_StdDev.png"))

In [10]:
# =============================================================================
# 4. MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":
    print("Starting generation of final 13 plots for the thesis...")
    all_datasets_history = {}

    for ds in DATASETS_LIST:
        print(f"-> Processing dataset: {ds}...")
        history_dict, final_df = load_dataset_data(ds)
        
        if not final_df.empty:
            all_datasets_history[ds] = history_dict
            
            # 1. Boxplots RMSE (3 Plots)
            plot_custom_rmse_boxplots(final_df, ds)
            # 2. Boxplots Size (3 Plots)
            plot_custom_size_boxplots(final_df, ds)
            
            # 3. RMSE Evolution (3 Plots)
            plot_custom_evolution(history_dict, ds, metric_type='RMSE')
            # 4. Size Evolution (3 Plots)
            plot_custom_evolution(history_dict, ds, metric_type='Size')

    # 5. Global Std Dev (1 Plot)
    if all_datasets_history:
        print("-> Generating global standard deviation plot...")
        plot_global_std_dev(all_datasets_history)

    print("Final plots generated successfully and saved in the plots directory.")

Starting generation of final 13 plots for the thesis...
-> Processing dataset: Boston...
-> Processing dataset: Cooling...
-> Processing dataset: Toxicity...
-> Generating global standard deviation plot...
Final plots generated successfully and saved in the plots directory.
